# DiCo-NLI runner (Kaggle): thí nghiệm A, diff input

Luật: PLAN.md mục 2d, "Thí nghiệm A" (chạy trên Kaggle: ngưỡng go OOF mean3 ≥ 0.8644, baseline mDeBERTa Kaggle 0.8614).
Config: `m0_mdeberta_xnli_pair_all` + `diff_input=true max_len=96`, tên run `m0_mdeberta_xnli_pair_all_diff`.

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**. *Save Version → Save & Run All*. Ước tính 40–60 phút.

- GPU 0: 3 seed. GPU 1: OOF 3 fold. Sau đó chọn bias NEG trên OOF và chấm dev.

Kết quả: tải `runs_diff.zip` ở tab *Output*, bỏ vào `kaggle/`.

In [ ]:
REPO_URL = 'https://github.com/danh-tc/DiCo-NLI.git'
!git clone --depth 1 $REPO_URL /kaggle/working/dico-nli
%cd /kaggle/working/dico-nli
!git log -1 --oneline
!pip install -q -e .
!bash scripts/setup_data.sh
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch', torch.__version__, 'cuda', torch.version.cuda)
assert torch.cuda.is_available(), 'Không có CUDA: bật GPU T4 x2 trong Settings'

In [ ]:
%%bash
# Smoke (~1 phút): 0.02 epoch, cũng là bước tải model
set -e
CUDA_VISIBLE_DEVICES=0 PYTHONPATH=src python -m dico.train --config configs/m0_mdeberta_xnli_pair_all.yaml --seed 1 --set name=smoke epochs=0.02 diff_input=true max_len=96 "decoders=[argmax,self_swap]" > /kaggle/working/smoke.log 2>&1 || { tail -40 /kaggle/working/smoke.log; exit 1; }
rm -rf /kaggle/working/runs/smoke
echo smoke OK

In [ ]:
%%bash
export SUFFIX=_diff EXTRA="diff_input=true max_len=96"
bash scripts/kaggle_queue.sh 0 train:main:1 train:main:2 train:main:3 &
bash scripts/kaggle_queue.sh 1 oof:main &
wait
cat /kaggle/working/runs/logs/status.txt

In [ ]:
%%bash
# Chọn bias NEG trên OOF (mean3), chấm dev 3 seed tại bias đó
N=m0_mdeberta_xnli_pair_all_diff
PYTHONPATH=src python scripts/tune_neg_bias.py --config configs/m0_mdeberta_xnli_pair_all.yaml \
  --oof /kaggle/working/runs/oof/$N --runs /kaggle/working/runs/$N \
  --set name=$N diff_input=true max_len=96 2>&1 | grep -v -i warn | tee /kaggle/working/runs/logs/tune_diff.log
grep FAIL /kaggle/working/runs/logs/status.txt || echo "no failures"

In [ ]:
!cd /kaggle/working && zip -qr runs_diff.zip runs -x '*/ckpt/*' '*/model/*' && ls -lh runs_diff.zip